<a href="https://colab.research.google.com/github/JZjj/llm-sys-project/blob/main/YJQ/Single_Model_one_shot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Library

# Data Loading

In [ ]:
import os
import subprocess
from pathlib import Path

CASTLE_REPO = "https://github.com/CASTLE-Benchmark/CASTLE-Benchmark.git"
# Fix: Use a path relative to the current working directory in Colab
DATA_DIR = Path("/content") / "data" / "CASTLE-Benchmark"

def clone_if_needed():
    if DATA_DIR.exists():
        print("CASTLE already cloned at", DATA_DIR)
        return
    DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", CASTLE_REPO, str(DATA_DIR)], check=True)
    print("Cloned CASTLE to", DATA_DIR)

def list_cases():
    # CASTLE repo structure: each test is under e.g. tests/... (repo may change — inspect after cloning)
    # Here we do a simple walk to collect .c/.h files
    file_list = []
    for p in DATA_DIR.rglob("*.c"):
        file_list.append(str(p))
    for p in DATA_DIR.rglob("*.cpp"):
        file_list.append(str(p))
    print(f"Found {len(file_list)} source files")
    return file_list

if __name__ == "__main__":
    clone_if_needed()
    files = list_cases()
    # Optionally save listing
    out = DATA_DIR.parent / "file_listing.txt"
    out.write_text("\n".join(files))
    print("Wrote listing to", out)


Cloned CASTLE to /content/data/CASTLE-Benchmark
Found 250 source files
Wrote listing to /content/data/file_listing.txt


# LLM Model Selection

# groq_API

- Free plan [rate limits](https://console.groq.com/docs/rate-limits?utm_source=chatgpt.com)

- Model selection

> RPM: Requests per minute
>
> RPD: Requests per day
>
> TPM: Tokens per minute
>
> TPD: Tokens per day

|Model Name |RPM | 	RPD| 	TPM	|TPD| Advantage|
|---|---|---|---|---|---|
|Llama 4 Scout 17B Instruct|30|1K|30K|500K|Specifically trained for reasoning and code safety awareness|
|openai/gpt-oss-20b|30|1K|8K|200K|High code quality, Strong debugging ability ,Fast reasoning (very fast on Groq)|
|llama-3.1-8b-instant|30|14.4K|6K|500K| Fast|

In [ ]:
# Colab get the corresponding api key from Secret
from google.colab import userdata
api_key=userdata.get("GROQ_API_KEY")

# Select model
Model_name="Llama 4 Scout 17B Instruct"

# GROQ

!pip install groq

from groq import Groq

client = Groq(
    api_key=api_key,
)

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Explain the importance of fast language models",
        }
    ],
    model=Model_name,
)

print(chat_completion.choices[0].message.content)

Fast language models are crucial in today's technology landscape, and their importance can be understood from several perspectives:

1. **Real-time Applications**: Fast language models enable real-time applications such as virtual assistants, chatbots, and live translation services. These models can quickly process and respond to user input, providing a seamless and interactive experience.
2. **Improved User Experience**: Fast language models can significantly enhance the user experience by reducing latency and increasing the speed of text generation, sentiment analysis, and language translation. This leads to higher user satisfaction and engagement.
3. **Efficient Processing**: Fast language models can process large volumes of text data quickly, making them ideal for applications such as text summarization, entity recognition, and topic modeling. This efficiency enables organizations to analyze and extract insights from large datasets in a shorter amount of time.
4. **Resource Optimiz

# Zero Shot

Ask an LLM to review and refine existing code to strengthen its security posture.

Prompt:

"You are an expert C programmer specializing in secure coding practices.
Please review the following C code and identify any potential security vulnerabilities, such as buffer overflows, format string bugs, integer overflows, memory leaks, or improper input validation.
Rewrite the code to eliminate these vulnerabilities, ensuring the improved code is robust, functional, and follows secure coding best practices.
Only provide the improved C code block, enclosed in triple backticks (```c\n...\n```), and nothing else."


In [ ]:
import os
from pathlib import Path
import re

def extract_code_from_response(text: str, language: str = "c") -> str:
    """
    Extracts a code block of a specified language from a given text.
    Assumes the code is within ```<language>\n...\n``` block.
    """
    # Pattern to match ```<language>\n...\n``` block
    pattern = rf"```(?:{language}\\n|{language}\\r\\n|{language}\\s*)?([\\s\\S]*?)```"
    m = re.search(pattern, text)
    if m:
        return m.group(1).strip()
    # Fallback: if no language-specific block, try to find any generic code block ```\n...\n```
    m_general = re.search(r"```(?:\\n|\\r\\n|\\s*)?([\\s\\S]*?)```", text)
    if m_general:
        return m_general.group(1).strip()
    # Fallback: if no code block markers, return the entire text (might not be ideal)
    return text.strip()

# Path to the C file to be improved
c_file_path = "/content/data/CASTLE-Benchmark/datasets/CASTLE-C250/CASTLE-125-1.c"

# Read the content of the C file
try:
    c_code_original = Path(c_file_path).read_text()
    print(f"Successfully read C code from: {c_file_path}")
except FileNotFoundError:
    print(f"Error: C file not found at {c_file_path}")
    c_code_original = "// Error: File not found. Please check the path.\nint main() { return 1; }"
except Exception as e:
    print(f"Error reading C file: {e}")
    c_code_original = f"// Error reading file: {e}\nint main() {{ return 1; }}"

# Construct the prompt for the LLM
prompt_content = f"""You are an expert C programmer specializing in secure coding practices.
Please review the following C code and identify any potential security vulnerabilities, such as buffer overflows, format string bugs, integer overflows, memory leaks, or improper input validation.
Rewrite the code to eliminate these vulnerabilities, ensuring the improved code is robust, functional, and follows secure coding best practices.
Only provide the improved C code block, enclosed in triple backticks (```c\n...\n```), and nothing else.

```c
{c_code_original}
```
"""

messages = [
    {"role": "system", "content": "You are a helpful and experienced C programming assistant focused on security improvements."},
    {"role": "user", "content": prompt_content}
]

print("Sending request to LLM for code improvement...")

# Call the Groq API
try:
    chat_completion = client.chat.completions.create(
        messages=messages,
        model=Model_name,
        temperature=0.7 # Adjust temperature for desired creativity/consistency
    )

    llm_response_content = chat_completion.choices[0].message.content
    print("\nLLM response received. Attempting to extract improved C code...")

    # Extract the improved C code from the LLM's response
    improved_c_code = extract_code_from_response(llm_response_content, language="c")

    print("\n--- Original C Code ---")
    print(c_code_original)
    print("\n--- Improved C Code (by LLM) ---")
    print(improved_c_code)

    # Optionally, save the improved code to a new file
    output_dir = Path("/content/improved_c_code")
    output_dir.mkdir(parents=True, exist_ok=True)
    output_file_path = output_dir / "CASTLE-125-1_improved.c"
    output_file_path.write_text(improved_c_code)
    print(f"\nImproved code saved to: {output_file_path}")

except Exception as e:
    print(f"An error occurred during LLM interaction: {e}")
    print("Please ensure your API key is correct and the model is available.")
    print(f"Raw LLM response (if available): {llm_response_content if 'llm_response_content' in locals() else 'N/A'}")


Successfully read C code from: /content/data/CASTLE-Benchmark/datasets/CASTLE-C250/CASTLE-125-1.c
Sending request to LLM for code improvement...

LLM response received. Attempting to extract improved C code...

--- Original C Code ---
#include <stdio.h>

int main() {
    int array[5] = {1, 2, 3, 4, 5};

    for(int i=0; i<=5; i++) {
        printf("%d\n", array[i]);
    }

    return 0;
}

--- Improved C Code (by LLM) ---
```c
#include <stdio.h>
#include <stdint.h>

int main() {
    int32_t array[5] = {1, 2, 3, 4, 5};
    const uint32_t array_size = sizeof(array) / sizeof(array[0]);

    for (uint32_t i = 0; i < array_size; i++) {
        printf("%" PRId32 "\n", array[i]);
    }

    return 0;
}
```

Improved code saved to: /content/improved_c_code/CASTLE-125-1_improved.c


Read all code file paths listed in `file_listing.txt`, send each file's content to the LLM for improvement, and save all improved code outputs to the `improved_c_code` directory.

For testing, we only improve the codes in the first two files.

In [ ]:
import os
from pathlib import Path
import re

# -----------------------------
# Extract code from LLM response
# -----------------------------
def extract_code_from_response(text: str, language: str = "c") -> str:
    pattern = rf"```(?:{language}\\n|{language}\\r\\n|{language}\\s*)?([\\s\\S]*?)```"
    m = re.search(pattern, text)
    if m:
        return m.group(1).strip()

    m_general = re.search(r"```(?:\\n|\\r\\n|\\s*)?([\\s\\S]*?)```", text)
    if m_general:
        return m_general.group(1).strip()

    return text.strip()

# -----------------------------
# Load all file paths
# -----------------------------
file_list_path = "/content/data/file_listing.txt"

if not Path(file_list_path).exists():
    raise FileNotFoundError("file_listing.txt not found!")

with open(file_list_path, "r") as f:
    c_files = [line.strip() for line in f if line.strip()]

print(f"Found {len(c_files)} C files to process.")

# -----------------------------
# Where to save improved code
# -----------------------------
output_dir = Path("/content/improved_c_code")
output_dir.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Process each file
# -----------------------------
for c_file_path in c_files[:2]:
    print(f"\n==============================")
    print(f"Processing: {c_file_path}")
    print("==============================")

    # 1. Read C code
    c_file_path = Path(c_file_path)
    try:
        c_code_original = c_file_path.read_text()
        print(f"Successfully read: {c_file_path}")
    except Exception as e:
        print(f"Error reading file: {e}")
        continue

    # 2. Construct LLM prompt
    prompt_content = f"""You are an expert C programmer specializing in secure coding practices.
    Please review the following C code and identify any potential security vulnerabilities, such as buffer overflows, format string bugs, integer overflows, memory leaks, or improper input validation.
    Rewrite the code to eliminate these vulnerabilities, ensuring the improved code is robust, functional, and follows secure coding best practices.
    Only provide the improved C code block, enclosed in triple backticks (```c\\n...\\n```), and nothing else.

    ```c
    {c_code_original}
    """
    messages = [
        {"role": "system", "content": "You are a helpful and experienced C programming assistant focused on security improvements."},
        {"role": "user", "content": prompt_content}
    ]

    print("Sending request to LLM...")

    try:
        chat_completion = client.chat.completions.create(
            messages=messages,
            model=Model_name,
            temperature=0.7
        )
        llm_response_content = chat_completion.choices[0].message.content

        # 3. Extract code
        improved_c_code = extract_code_from_response(llm_response_content, language="c")

        # 4. Save result
        out_name = c_file_path.stem + "_improved.c"
        out_path = output_dir / out_name
        out_path.write_text(improved_c_code)

        print(f"Saved improved file to: {out_path}")

    except Exception as e:
        print(f"Error during LLM call: {e}")
        print(f"Raw response: {llm_response_content if 'llm_response_content' in locals() else 'N/A'}")

Found 250 C files to process.

Processing: /content/data/CASTLE-Benchmark/datasets/CASTLE-C250/CASTLE-415-1.c
Successfully read: /content/data/CASTLE-Benchmark/datasets/CASTLE-C250/CASTLE-415-1.c
Sending request to LLM...
Saved improved file to: /content/improved_c_code/CASTLE-415-1_improved.c

Processing: /content/data/CASTLE-Benchmark/datasets/CASTLE-C250/CASTLE-843-2.c
Successfully read: /content/data/CASTLE-Benchmark/datasets/CASTLE-C250/CASTLE-843-2.c
Sending request to LLM...
Saved improved file to: /content/improved_c_code/CASTLE-843-2_improved.c


# Few Shot

# Evaluation-metric-based prompting